In [ ]:
# ==============================================================================
# NEXUS OptiML 2026 — Operation Insight
# Complete Production Pipeline:
# 1. Forensic Cleaning & Physical/Accounting Imputations
# 2. High-Signal Domain Features (Bracketing, Landed Cost, Stationary Macro)
# 3. Stratified Group Target Encoding (order_id-grouped)
# 4. Multi-Model GPU Suite:
#    - Model 1: GPU Random Forest (XGBRFClassifier on CUDA)
#    - Model 2: GPU LightGBM (Optuna-Tuned on Temporal Holdout)
#    - Model 3: GPU XGBoost (Histogram CUDA)
#    - Model 4: GPU CatBoost (CUDA Native)
# 5. GPU Optuna-Optimized Percentile Rank Ensemble & Verified Submission
# ==============================================================================

import os
import re
import gc
import difflib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
from catboost.utils import get_gpu_device_count
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ==============================================================================
# CELL 0: HARDWARE PROBE & GPU CONFIGURATION
# ==============================================================================
def configure_gpu_backends():
    torch_gpu = torch.cuda.is_available()
    torch_device = torch.device("cuda" if torch_gpu else "cpu")

    try:
        cat_gpu = get_gpu_device_count() > 0
    except Exception:
        cat_gpu = False

    xgb_device = "cpu"
    try:
        _X = np.random.randn(20, 2).astype(np.float32)
        _y = np.random.randint(0, 2, size=20)
        _m = xgb.XGBClassifier(n_estimators=1, tree_method="hist", device="cuda")
        _m.fit(_X, _y)
        xgb_device = "cuda"
    except Exception:
        xgb_device = "cpu"

    lgb_gpu_params = {"n_jobs": -1}
    for dev in ["cuda", "gpu"]:
        try:
            _X = np.random.randn(30, 2).astype(np.float32)
            _y = np.random.randint(0, 2, size=30)
            _m = lgb.LGBMClassifier(n_estimators=2, device=dev, max_bin=255, verbose=-1)
            _m.fit(_X, _y)
            lgb_gpu_params = {"device": dev, "max_bin": 255, "n_jobs": -1}
            break
        except Exception:
            continue

    print("=" * 72)
    print(f"PyTorch Device  : {torch_device} ({torch.cuda.get_device_name(0) if torch_gpu else 'CPU'})")
    print(f"XGBoost / RF    : {xgb_device.upper()} (GPU Hist Engine)")
    print(f"CatBoost Device : {'GPU' if cat_gpu else 'CPU'}")
    print(f"LightGBM Params : {lgb_gpu_params}")
    print("=" * 72)
    return torch_device, xgb_device, cat_gpu, lgb_gpu_params

TORCH_DEVICE, XGB_DEVICE, CAT_GPU, LGB_GPU_PARAMS = configure_gpu_backends()


# ==============================================================================
# CELL 1: DATA LOADING & PATH RESOLUTION
# ==============================================================================
DATA_DIR = Path(".")
if not (DATA_DIR / "test.csv").exists():
    for root, _, files in os.walk("/kaggle/input"):
        if "test.csv" in files:
            DATA_DIR = Path(root)
            break

print(f"Working Directory: {DATA_DIR.resolve()}")

stores_raw = pd.read_csv(DATA_DIR / "stores.csv")
calendar_raw = pd.read_csv(DATA_DIR / "calendar.csv")
customers_raw = pd.read_csv(DATA_DIR / "customers.csv")
products_raw = pd.read_csv(DATA_DIR / "products.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")
sample_sub_raw = pd.read_csv(DATA_DIR / "sample_submission.csv")

train_path = DATA_DIR / "train.csv"
if train_path.exists():
    train_raw = pd.read_csv(train_path)
    print(f"Loaded train.csv : {train_raw.shape}")
else:
    print("WARNING: train.csv not found in directory. Creating schema scaffold for verification...")
    train_raw = test_raw.sample(n=min(120_000, len(test_raw)), random_state=SEED).copy()
    train_raw["order_ts"] = pd.to_datetime(train_raw["order_ts"]) - pd.Timedelta(days=180)
    train_raw["order_ts"] = train_raw["order_ts"].dt.strftime("%Y-%m-%d %H:%M:%S")
    train_raw["returned"] = np.random.binomial(1, 0.22, size=len(train_raw))

print(f"Loaded test.csv  : {test_raw.shape}")


# ==============================================================================
# CELL 2: CANONICAL CLEANING & DEDUPLICATION OF AUXILIARY TABLES
# ==============================================================================
def clean_mapping(dirty_series, canonical_list):
    norm_dict = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in canonical_list}
    norm_keys = list(norm_dict.keys())
    mapper = {}
    for val in dirty_series.dropna().unique():
        k = re.sub(r"[^a-z0-9]", "", str(val).lower())
        if k in norm_dict:
            mapper[val] = norm_dict[k]
        else:
            match = difflib.get_close_matches(k, norm_keys, n=1, cutoff=0.6)
            mapper[val] = norm_dict[match[0]] if match else str(val).strip()
    return mapper

# 1. Clean stores.csv
CANON_REGIONS = sorted(stores_raw["region"].dropna().unique().tolist())
stores_df = stores_raw.copy()
stores_df["is_virtual_store"] = stores_df["size_sqm"].isna().astype(np.int8)
format_median_sqm = stores_df.groupby("format", sort=False)["size_sqm"].transform("median")
stores_df["size_sqm"] = stores_df["size_sqm"].fillna(format_median_sqm).fillna(0.0).astype(np.float32)
stores_df["opened_date"] = pd.to_datetime(stores_df["opened_date"], errors="coerce")
stores_df = stores_df.rename(columns={"channel": "store_channel", "region": "store_region"})

# 2. Clean & Deduplicate customers.csv
cust_df = customers_raw.copy()
cust_region_map = clean_mapping(cust_df["region"], CANON_REGIONS)
cust_df["cust_region"] = cust_df["region"].map(cust_region_map).fillna("Unknown")
cust_df.drop(columns=["region"], inplace=True)
cust_df["signup_date"] = pd.to_datetime(cust_df["signup_date"], errors="coerce")
cust_df["record_updated_at"] = pd.to_datetime(cust_df["record_updated_at"], errors="coerce")

cust_df = cust_df.sort_values(["customer_id", "record_updated_at"])
cust_df["age_band"] = cust_df["age_band"].fillna(
    cust_df.groupby("customer_id", sort=False)["age_band"].transform("last")
).fillna(
    cust_df.groupby("customer_id", sort=False)["age_band"].transform("first")
)

cust_stats = cust_df.groupby("customer_id", sort=False).agg(
    cust_profile_updates=("record_updated_at", "count"),
    cust_loyalty_nunique=("loyalty_tier", "nunique"),
).reset_index()
cust_stats["has_loyalty_tier_changed"] = (cust_stats["cust_loyalty_nunique"] > 1).astype(np.int8)

cust_df = cust_df.drop_duplicates(subset=["customer_id"], keep="last").copy()
cust_df = cust_df.merge(cust_stats[["customer_id", "cust_profile_updates", "has_loyalty_tier_changed"]], on="customer_id", how="left")

cust_df["age_band"] = cust_df["age_band"].fillna(
    cust_df.groupby(["customer_type", "loyalty_tier"], sort=False)["age_band"].transform(
        lambda s: s.mode().iloc[0] if not s.mode().empty else "25-34"
    )
).fillna("25-34")

age_midpoints = {"18-24": 21.0, "25-34": 29.5, "35-44": 39.5, "45-54": 49.5, "55-64": 59.5, "65+": 70.0}
cust_df["cust_age_est"] = cust_df["age_band"].map(age_midpoints).fillna(35.0).astype(np.float32)

loyalty_ranks = {"bronze": 1, "silver": 2, "gold": 3, "platinum": 4}
cust_df["loyalty_rank"] = cust_df["loyalty_tier"].map(loyalty_ranks).fillna(1).astype(np.int8)

# 3. Clean & Physical-Impute products.csv
prod_df = products_raw.copy()
CANON_CATS = [
    "home_decor", "kitchen_dining", "sports_leisure", "toys_gifts",
    "fashion_accessories", "office_supplies", "health_beauty", "technology",
    "electronics_accessories", "garden_diy", "furniture", "food_beverages"
]
prod_cat_map = clean_mapping(prod_df["category"], CANON_CATS)
prod_df["category"] = prod_df["category"].map(prod_cat_map).fillna("home_decor")
prod_df["launch_date"] = pd.to_datetime(prod_df["launch_date"], errors="coerce")

prod_df["prod_volume_cm3"] = (
    prod_df["length_cm"].clip(lower=1.0) * prod_df["width_cm"].clip(lower=1.0) * prod_df["height_cm"].clip(lower=1.0)
).astype(np.float32)
prod_df["prod_surface_area_cm2"] = (
    2.0 * (prod_df["length_cm"] * prod_df["width_cm"] + prod_df["length_cm"] * prod_df["height_cm"] + prod_df["width_cm"] * prod_df["height_cm"])
).astype(np.float32)
prod_df["prod_max_dim_cm"] = prod_df[["length_cm", "width_cm", "height_cm"]].max(axis=1).astype(np.float32)
prod_df["prod_min_dim_cm"] = prod_df[["length_cm", "width_cm", "height_cm"]].min(axis=1).astype(np.float32)
prod_df["prod_aspect_ratio"] = (prod_df["prod_max_dim_cm"] / (prod_df["prod_min_dim_cm"] + 1e-3)).astype(np.float32)

prod_df["weight_missing"] = (prod_df["weight_kg"].isna() | (prod_df["weight_kg"] <= 0)).astype(np.int8)
prod_df.loc[prod_df["weight_kg"] <= 0, "weight_kg"] = np.nan
prod_df["density_raw"] = prod_df["weight_kg"] / prod_df["prod_volume_cm3"]

cat_density = prod_df.groupby(["category", "packaging_type"], sort=False)["density_raw"].transform("median")
prod_df["weight_kg"] = prod_df["weight_kg"].fillna(
    (prod_df["prod_volume_cm3"] * cat_density.fillna(prod_df["density_raw"].median())).clip(lower=0.05, upper=45.0)
).astype(np.float32)
prod_df["prod_density"] = (prod_df["weight_kg"] / prod_df["prod_volume_cm3"]).astype(np.float32)
prod_df["prod_price_per_kg"] = (prod_df["list_price"] / (prod_df["weight_kg"] + 0.01)).astype(np.float32)
prod_df.drop(columns=["density_raw"], inplace=True)

# 4. Stationary Macro Transformations on calendar.csv (Anti-Drift)
cal_df = calendar_raw.copy()
cal_df["week_start"] = pd.to_datetime(cal_df["week_start"])
cal_df = cal_df.sort_values("week_start").reset_index(drop=True)

for col in ["cpi", "fuel_price", "temperature_f", "unemployment", "fuel_surcharge"]:
    cal_df[f"{col}_diff_1w"] = cal_df[col].diff(1).fillna(0.0).astype(np.float32)
    lag4 = cal_df[col].shift(4)
    cal_df[f"{col}_pct_4w"] = ((cal_df[col] - lag4) / (lag4.abs() + 1e-4)).fillna(0.0).astype(np.float32)
    r_mean = cal_df[col].rolling(8, min_periods=1).mean()
    r_std = cal_df[col].rolling(8, min_periods=1).std().fillna(1.0) + 1e-4
    cal_df[f"{col}_zscore_8w"] = ((cal_df[col] - r_mean) / r_std).astype(np.float32)

cal_df["rolling_holiday_3w"] = cal_df["holiday_flag"].rolling(3, center=True, min_periods=1).mean().astype(np.float32)

# Drop raw non-stationary levels to eliminate train-to-test drift
cal_df.drop(columns=["cpi", "unemployment", "fuel_price", "temperature_f"], inplace=True)
print("Auxiliary tables cleaned and transformed with stationary macroeconomic metrics.")


# ==============================================================================
# CELL 3: RELATIONAL IMPUTATION & HIGH-SIGNAL DOMAIN FEATURE ENGINEERING
# ==============================================================================
n_train = len(train_raw)
y_train = train_raw["returned"].values.astype(np.int8)

df = pd.concat([train_raw.drop(columns=["returned"]), test_raw], axis=0, ignore_index=True)
del train_raw, test_raw
gc.collect()

print(f"\nProcessing combined transaction table: {df.shape}")

df["order_dt"] = pd.to_datetime(df["order_ts"], errors="coerce")
df["week_start"] = df["order_dt"].dt.to_period("W-SUN").dt.start_time

# Relational merges
df = df.merge(cust_df, on="customer_id", how="left")
df = df.merge(prod_df, on="product_id", how="left")
df = df.merge(stores_df, on="store_id", how="left")
df = pd.merge_asof(
    df.sort_values("week_start"),
    cal_df.sort_values("week_start"),
    on="week_start",
    direction="nearest"
).sort_index()

del cust_df, prod_df, stores_df, cal_df
gc.collect()

# --- 3A. Fix the 2% 100x unit_price Cents Anomaly ---
ratio_list = df["unit_price"] / (df["list_price"] + 1e-4)
ratio_val = (df["unit_price"] * df["quantity"]) / (df["order_value"] + 1.0)
cents_mask = (ratio_list > 10.0) & (ratio_val > 15.0)

df["is_unit_price_cents_glitch"] = cents_mask.astype(np.int8)
df["unit_price"] = np.where(cents_mask, df["unit_price"] / 100.0, df["unit_price"]).astype(np.float32)
df["order_value"] = np.where(df["order_value"] <= 0, df["unit_price"] * df["quantity"], df["order_value"]).astype(np.float32)

# --- 3B. Exact Accounting Imputation for Missing Discount (r = 0.9737) ---
df["discount_was_null"] = df["discount"].isna().astype(np.int8)
implied_disc = (1.0 - (df["unit_price"] / (df["list_price"] + 1e-4))).clip(0.0, 0.80).astype(np.float32)
df["discount"] = df["discount"].fillna(implied_disc).astype(np.float32)
df["discount_discrepancy"] = (df["discount"] - implied_disc).astype(np.float32)

# --- 3C. Ultra-Fast Order-Level & Relational Missingness Imputation ---
for col in ["tz_hint", "payment_method", "ship_mode", "session_clicks"]:
    df[f"{col}_was_null"] = df[col].isna().astype(np.int8)
    df[col] = df[col].fillna(df.groupby("order_id", sort=False)[col].transform("first"))

df.loc[df["ship_mode"].isna() & (df["channel"] == "store"), "ship_mode"] = "in_store"
df["ship_mode"] = df["ship_mode"].fillna(
    df.groupby(["channel", "return_window_days"], sort=False)["ship_mode"].transform(
        lambda s: s.mode().iloc[0] if not s.mode().empty else "standard"
    )
).fillna("standard")

df["payment_method"] = df["payment_method"].fillna(
    df.groupby(["channel", "device_type"], sort=False)["payment_method"].transform(
        lambda s: s.mode().iloc[0] if not s.mode().empty else "card"
    )
).fillna("card")

df["tz_hint"] = df["tz_hint"].fillna(
    df.groupby("store_id", sort=False)["tz_hint"].transform("first")
).fillna("UTC")

dev_clicks_med = df.groupby(["device_type", "channel"], sort=False)["session_clicks"].transform("median")
df["session_clicks"] = df["session_clicks"].fillna(dev_clicks_med).fillna(9.0).astype(np.float32)

df["has_coupon"] = df["coupon_code"].notna().astype(np.int8)
df["coupon_code"] = df["coupon_code"].fillna("NO_COUPON")

# --- FIXED: Logistics Imputation of freight_cost (Resolves KeyError: 'Column not found: None') ---
df["line_weight_kg"] = (df["weight_kg"] * df["quantity"]).astype(np.float32)
df.loc[df["freight_cost"].isna() & (df["ship_mode"] == "in_store"), "freight_cost"] = 0.0

df["freight_rate_factor"] = df["freight_cost"] / ((df["line_weight_kg"] + 1.0) * (1.0 + df["fuel_surcharge"]))
grp_freight_rate = df.groupby(["ship_mode", "channel", "warehouse_zone"], sort=False)["freight_rate_factor"].transform("median")
imputed_freight = (grp_freight_rate * (df["line_weight_kg"] + 1.0) * (1.0 + df["fuel_surcharge"])).clip(lower=0.0)
df["freight_cost"] = df["freight_cost"].fillna(imputed_freight).fillna(df["freight_cost"].median()).astype(np.float32)
df.drop(columns=["freight_rate_factor"], inplace=True)

# --- 3D. High-Signal Domain & Behavioral Features ---
print("Extracting domain-specific return signals (bracketing, financial friction, landed cost)...")

# 1. Financial & Landed Cost Interactions
df["line_exact_revenue"] = (df["unit_price"] * df["quantity"]).astype(np.float32)
df["line_list_revenue"] = (df["list_price"] * df["quantity"]).astype(np.float32)
df["dollar_savings"] = (df["line_list_revenue"] - df["line_exact_revenue"]).clip(lower=0.0).astype(np.float32)
df["unit_to_list_ratio"] = (df["unit_price"] / (df["list_price"] + 1e-4)).astype(np.float32)
df["order_val_rounding_diff"] = (df["order_value"] - df["line_exact_revenue"]).astype(np.float32)
df["freight_to_order_val_ratio"] = (df["freight_cost"] / (df["order_value"] + 1.0)).astype(np.float32)
df["total_landed_cost"] = (df["order_value"] + df["freight_cost"]).astype(np.float32)
df["unit_landed_cost"] = (df["total_landed_cost"] / df["quantity"].clip(lower=1)).astype(np.float32)
df["is_free_shipping"] = (df["freight_cost"] == 0).astype(np.int8)

# 2. Return Window & Friction Ratios
df["return_window_to_freight"] = (df["return_window_days"] / (df["freight_cost"] + 1.0)).astype(np.float32)
df["return_window_x_discount"] = (df["return_window_days"] * df["discount"]).astype(np.float32)
df["is_lenient_return_policy"] = ((df["return_window_days"] >= 30) & (df["freight_cost"] == 0)).astype(np.int8)

# 3. Intra-Order Bracketing & Basket Topology (Wardrobing Predictors)
order_grp = df.groupby("order_id", sort=False)
df["order_observed_lines"] = order_grp["order_line_id"].transform("count").astype(np.int16)
df["order_total_spend"] = order_grp["order_value"].transform("sum").astype(np.float32)
df["order_total_qty"] = order_grp["quantity"].transform("sum").astype(np.int32)
df["order_max_unit_price"] = order_grp["unit_price"].transform("max").astype(np.float32)
df["order_mean_unit_price"] = order_grp["unit_price"].transform("mean").astype(np.float32)
df["order_nunique_cats"] = order_grp["category"].transform("nunique").astype(np.int16)

df["line_spend_share"] = (df["order_value"] / (df["order_total_spend"] + 1e-3)).astype(np.float32)
df["line_qty_share"] = (df["quantity"] / (df["order_total_qty"] + 1e-3)).astype(np.float32)
df["line_price_vs_order_max"] = (df["unit_price"] / (df["order_max_unit_price"] + 1e-3)).astype(np.float32)
df["line_pos_ratio"] = (df["line_no"] / df["items_in_order"].clip(lower=1)).astype(np.float32)

# Bracketing indicators: ordering multiple items of the same category or from the same supplier in one checkout
df["same_cat_count_in_order"] = df.groupby(["order_id", "category"], sort=False)["order_line_id"].transform("count").astype(np.int16)
df["is_same_category_bracketed"] = (df["same_cat_count_in_order"] > 1).astype(np.int8)

df["same_supp_count_in_order"] = df.groupby(["order_id", "supplier_code"], sort=False)["order_line_id"].transform("count").astype(np.int16)
df["is_same_supplier_bracketed"] = (df["same_supp_count_in_order"] > 1).astype(np.int8)

# 4. Customer & Store Geographic Alignment
df["is_same_region_cust_store"] = (df["cust_region"] == df["store_region"]).astype(np.int8)
df["cust_country"] = df["cust_region"].astype(str).str.split("-").str[0]
df["store_country"] = df["store_region"].astype(str).str.split("-").str[0]
df["is_same_country_cust_store"] = (df["cust_country"] == df["store_country"]).astype(np.int8)
df["is_same_channel_order_store"] = (df["channel"] == df["store_channel"]).astype(np.int8)

# 5. Customer Profile Aggregations (Strictly Target-Free)
cust_grp = df.groupby("customer_id", sort=False)
df["cust_total_lines"] = cust_grp["order_line_id"].transform("count").astype(np.int32)
df["cust_mean_order_val"] = cust_grp["order_value"].transform("mean").astype(np.float32)
df["cust_mean_discount"] = cust_grp["discount"].transform("mean").astype(np.float32)
df["order_val_vs_cust_mean"] = (df["order_value"] / (df["cust_mean_order_val"] + 1.0)).astype(np.float32)

# 6. Stationary Lifecycle & Temporal Features
df["order_hour"] = df["order_dt"].dt.hour.fillna(12).astype(np.int8)
df["order_dayofweek"] = df["order_dt"].dt.dayofweek.fillna(0).astype(np.int8)
df["order_month"] = df["order_dt"].dt.month.fillna(6).astype(np.int8)
df["is_weekend"] = (df["order_dayofweek"] >= 5).astype(np.int8)
df["is_late_night"] = df["order_hour"].isin([23, 0, 1, 2, 3, 4, 5]).astype(np.int8)

df["hour_sin"] = np.sin(2 * np.pi * df["order_hour"] / 24.0).astype(np.float32)
df["hour_cos"] = np.cos(2 * np.pi * df["order_hour"] / 24.0).astype(np.float32)
df["dow_sin"] = np.sin(2 * np.pi * df["order_dayofweek"] / 7.0).astype(np.float32)
df["dow_cos"] = np.cos(2 * np.pi * df["order_dayofweek"] / 7.0).astype(np.float32)
df["month_sin"] = np.sin(2 * np.pi * df["order_month"] / 12.0).astype(np.float32)
df["month_cos"] = np.cos(2 * np.pi * df["order_month"] / 12.0).astype(np.float32)

df["cust_tenure_days"] = (df["order_dt"] - df["signup_date"]).dt.days.fillna(365).clip(lower=-30, upper=5000).astype(np.float32)
df["prod_age_days"] = (df["order_dt"] - df["launch_date"]).dt.days.fillna(365).clip(lower=0, upper=5000).astype(np.float32)
df["store_age_days"] = (df["order_dt"] - df["opened_date"]).dt.days.fillna(1000).clip(lower=0, upper=10000).astype(np.float32)
df["reviews_per_prod_year"] = (df["review_count"] / ((df["prod_age_days"] / 365.25) + 0.1)).astype(np.float32)
df["clicks_per_item"] = (df["session_clicks"] / df["items_in_order"].clip(lower=1)).astype(np.float32)

# Save reference IDs and timestamps
order_ts_train = df["order_dt"].iloc[:n_train].copy()
order_ids_train = df["order_id"].iloc[:n_train].values
submission_ids = df["order_line_id"].iloc[n_train:].values

RAW_DROP_COLS = [
    "order_line_id", "order_id", "order_ts", "order_dt", "week_start",
    "signup_date", "record_updated_at", "launch_date", "opened_date"
]
df.drop(columns=[c for c in RAW_DROP_COLS if c in df.columns], inplace=True)


# ==============================================================================
# CELL 4: SAFE ENCODING (NO ARBITRARY ORDINAL IDS IN TREE MODELS)
# ==============================================================================
SPLIT_CAT_COLS = [
    "channel", "payment_method", "ship_mode", "promo_channel_code",
    "warehouse_zone", "picker_shift", "device_type", "cust_region",
    "customer_type", "age_band", "preferred_contact", "loyalty_tier",
    "category", "packaging_type", "color", "store_channel", "store_region", "format",
    "tz_hint", "cust_country", "store_country"
]
SPLIT_CAT_COLS = [c for c in SPLIT_CAT_COLS if c in df.columns]

# High-cardinality IDs that MUST NOT be arbitrarily ordinal-split by decision trees
HIGH_CARD_ID_COLS = ["customer_id", "product_id", "store_id", "supplier_code", "coupon_code"]

print(f"\nEncoding {len(SPLIT_CAT_COLS)} categorical features with OrdinalEncoder...")
ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.int32)
df[SPLIT_CAT_COLS] = ord_enc.fit_transform(df[SPLIT_CAT_COLS].astype(str))

# Frequency Encoding across the entire interaction graph
for c in HIGH_CARD_ID_COLS:
    freq = df[c].value_counts()
    df[f"{c}_freq"] = df[c].map(freq).fillna(0).astype(np.float32)

X_train_df = df.iloc[:n_train].copy().reset_index(drop=True)
X_test_df = df.iloc[n_train:].copy().reset_index(drop=True)
del df
gc.collect()

# Stratified Group K-Fold Out-of-Fold Target Encoding (strictly grouped by order_id)
N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS)
folds = list(sgkf.split(X_train_df, y_train, groups=order_ids_train))

TE_COLS = ["product_id", "store_id", "supplier_code", "category", "cust_region", "channel", "ship_mode", "return_window_days"]
TE_COLS = [c for c in TE_COLS if c in X_train_df.columns]
global_mean = float(y_train.mean())
SMOOTHING = 25.0

print(f"Generating leak-free Out-Of-Fold Target Encodings across {len(TE_COLS)} attributes...")
for col in TE_COLS:
    oof_te = np.full(n_train, global_mean, dtype=np.float32)
    test_te_accum = np.zeros(len(X_test_df), dtype=np.float32)

    for tr_idx, val_idx in folds:
        tr_s = X_train_df[col].iloc[tr_idx]
        tr_y = pd.Series(y_train[tr_idx], index=tr_s.index)
        stats = tr_y.groupby(tr_s, sort=False).agg(["count", "mean"])
        smooth = (stats["count"] * stats["mean"] + SMOOTHING * global_mean) / (stats["count"] + SMOOTHING)

        oof_te[val_idx] = X_train_df[col].iloc[val_idx].map(smooth).fillna(global_mean).values
        test_te_accum += X_test_df[col].map(smooth).fillna(global_mean).values / N_SPLITS

    X_train_df[f"{col}_te"] = oof_te
    X_test_df[f"{col}_te"] = test_te_accum

# Drop raw high-cardinality IDs so trees do not memorize them
X_train_df.drop(columns=[c for c in HIGH_CARD_ID_COLS if c in X_train_df.columns], inplace=True)
X_test_df.drop(columns=[c for c in HIGH_CARD_ID_COLS if c in X_test_df.columns], inplace=True)

# Clean infs/nans and downcast to float32
X_train_df = X_train_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
X_test_df = X_test_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

FEATURES = list(X_train_df.columns)
print(f"Final Feature Matrix: {len(FEATURES)} features across {n_train:,} training lines.")

X_train_np = np.ascontiguousarray(X_train_df.to_numpy(dtype=np.float32))
X_test_np = np.ascontiguousarray(X_test_df.to_numpy(dtype=np.float32))
del X_train_df, X_test_df
gc.collect()


# ==============================================================================
# CELL 5: TEMPORAL HOLDOUT SET & FAST OPTUNA TUNING
# ==============================================================================
min_ts = order_ts_train.min()
max_ts = order_ts_train.max()
months_elapsed = ((order_ts_train - min_ts).dt.days / 30.4375).fillna(12.0).to_numpy(dtype=np.float32)

# Weight recent transactions slightly higher to generalize into Months 25-30
sample_weights = np.exp(0.02 * months_elapsed).astype(np.float32)

# Last 100 days of train data acts as strict Temporal Validation holdout
temporal_cutoff = max_ts - pd.Timedelta(days=100)
temp_tr_mask = (order_ts_train < temporal_cutoff).to_numpy()
temp_val_mask = (order_ts_train >= temporal_cutoff).to_numpy()

print(f"Temporal Train (Months 1–20) : {temp_tr_mask.sum():,} lines")
print(f"Temporal Valid (Months 21–24): {temp_val_mask.sum():,} lines")

tr_sub_idx = np.where(temp_tr_mask)[0]
val_sub_idx = np.where(temp_val_mask)[0]
if len(tr_sub_idx) > 350_000:
    tr_sub_idx = np.random.choice(tr_sub_idx, size=350_000, replace=False)

def tune_gpu_lightgbm(n_trials=8):
    print(f"\n--- Running GPU Optuna Tuning on Temporal Holdout ({n_trials} trials) ---")
    X_tr, y_tr = X_train_np[tr_sub_idx], y_train[tr_sub_idx]
    X_va, y_va = X_train_np[val_sub_idx], y_train[val_sub_idx]

    def objective(trial):
        params = {
            "objective": "binary",
            "metric": "auc",
            "boosting_type": "gbdt",
            "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.08, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 45, 127),
            "max_depth": trial.suggest_int("max_depth", 6, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 50, 300),
            "subsample": trial.suggest_float("subsample", 0.65, 0.95),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.85),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 15.0, log=True),
            "n_estimators": 1200,
            "random_state": SEED,
            "verbose": -1,
            **LGB_GPU_PARAMS
        }
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(40, verbose=False)]
        )
        preds = model.predict_proba(X_va)[:, 1]
        return roc_auc_score(y_va, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, timeout=400)
    print(f"Best Temporal Holdout ROC-AUC: {study.best_value:.5f}")
    return study.best_params

best_lgb_params = tune_gpu_lightgbm(n_trials=8)


# ==============================================================================
# CELL 6: MULTI-MODEL GPU TRAINING (GPU RF, LIGHTGBM, XGBOOST, CATBOOST)
# ==============================================================================
oof_rf = np.zeros(n_train, dtype=np.float32)
pred_rf = np.zeros(len(X_test_np), dtype=np.float32)

oof_lgb = np.zeros(n_train, dtype=np.float32)
pred_lgb = np.zeros(len(X_test_np), dtype=np.float32)

oof_xgb = np.zeros(n_train, dtype=np.float32)
pred_xgb = np.zeros(len(X_test_np), dtype=np.float32)

oof_cat = np.zeros(n_train, dtype=np.float32)
pred_cat = np.zeros(len(X_test_np), dtype=np.float32)

lgb_final_cfg = {
    **best_lgb_params,
    "objective": "binary",
    "metric": "auc",
    "n_estimators": 2500,
    "subsample_freq": 1,
    "verbose": -1,
    **LGB_GPU_PARAMS
}

xgb_final_cfg = {
    "n_estimators": 2500,
    "learning_rate": 0.04,
    "max_depth": 7,
    "min_child_weight": 40,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 2.0,
    "reg_lambda": 6.0,
    "eval_metric": "auc",
    "tree_method": "hist",
    "device": XGB_DEVICE,
    "max_bin": 256,
    "early_stopping_rounds": 60,
    "n_jobs": -1
}

cat_final_cfg = {
    "iterations": 2500,
    "learning_rate": 0.05,
    "depth": 7,
    "l2_leaf_reg": 8.0,
    "eval_metric": "AUC",
    "task_type": "GPU" if CAT_GPU else "CPU",
    "border_count": 254,
    "random_seed": SEED,
    "verbose": 0,
    "early_stopping_rounds": 60
}

print("\n" + "=" * 72)
print("STARTING 5-FOLD GPU TRAINING ACROSS 4 DIVERSE ARCHITECTURES")
print("=" * 72)

for fold, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n--- Fold {fold + 1}/{N_SPLITS} ---")
    X_f_tr, y_f_tr = X_train_np[tr_idx], y_train[tr_idx]
    w_f_tr = sample_weights[tr_idx]
    X_f_val, y_f_val = X_train_np[val_idx], y_train[val_idx]

    # --- 1. GPU Random Forest (XGBRFClassifier) ---
    rf_gpu = xgb.XGBRFClassifier(
        n_estimators=300,
        max_depth=12,
        learning_rate=1.0,
        subsample=0.8,
        colsample_bynode=0.25,
        min_child_weight=20,
        reg_lambda=2.0,
        tree_method="hist",
        device=XGB_DEVICE,
        max_bin=256,
        eval_metric="auc",
        random_state=SEED + fold,
        n_jobs=-1
    )
    rf_gpu.fit(X_f_tr, y_f_tr, sample_weight=w_f_tr)
    oof_rf[val_idx] = rf_gpu.predict_proba(X_f_val)[:, 1]
    pred_rf += rf_gpu.predict_proba(X_test_np)[:, 1] / N_SPLITS

    # --- 2. GPU LightGBM ---
    model_lgb = lgb.LGBMClassifier(**lgb_final_cfg, random_state=SEED + fold)
    model_lgb.fit(
        X_f_tr, y_f_tr,
        sample_weight=w_f_tr,
        eval_set=[(X_f_val, y_f_val)],
        callbacks=[lgb.early_stopping(60, verbose=False)]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_f_val)[:, 1]
    pred_lgb += model_lgb.predict_proba(X_test_np)[:, 1] / N_SPLITS

    # --- 3. GPU XGBoost ---
    model_xgb = xgb.XGBClassifier(**xgb_final_cfg, random_state=SEED + fold)
    model_xgb.fit(
        X_f_tr, y_f_tr,
        sample_weight=w_f_tr,
        eval_set=[(X_f_val, y_f_val)],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict_proba(X_f_val)[:, 1]
    pred_xgb += model_xgb.predict_proba(X_test_np)[:, 1] / N_SPLITS

    # --- 4. GPU CatBoost ---
    tr_pool = Pool(X_f_tr, y_f_tr, weight=w_f_tr)
    val_pool = Pool(X_f_val, y_f_val)
    model_cat = CatBoostClassifier(**cat_final_cfg)
    model_cat.fit(tr_pool, eval_set=val_pool, verbose=False)
    oof_cat[val_idx] = model_cat.predict_proba(val_pool)[:, 1]
    pred_cat += model_cat.predict_proba(Pool(X_test_np))[:, 1] / N_SPLITS

    auc_rf = roc_auc_score(y_f_val, oof_rf[val_idx])
    auc_lgb = roc_auc_score(y_f_val, oof_lgb[val_idx])
    auc_xgb = roc_auc_score(y_f_val, oof_xgb[val_idx])
    auc_cat = roc_auc_score(y_f_val, oof_cat[val_idx])

    print(f"Fold {fold + 1} OOF AUCs -> RF: {auc_rf:.5f} | LGB: {auc_lgb:.5f} | XGB: {auc_xgb:.5f} | CAT: {auc_cat:.5f}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 72)
print("FULL OUT-OF-FOLD ROC-AUC SUMMARY")
print(f"Model 1 [GPU Random Forest] : {roc_auc_score(y_train, oof_rf):.5f}")
print(f"Model 2 [GPU LightGBM]      : {roc_auc_score(y_train, oof_lgb):.5f}")
print(f"Model 3 [GPU XGBoost]       : {roc_auc_score(y_train, oof_xgb):.5f}")
print(f"Model 4 [GPU CatBoost]      : {roc_auc_score(y_train, oof_cat):.5f}")
print("=" * 72)


# ==============================================================================
# CELL 7: OPTUNA RANK-PERCENTILE META-ENSEMBLE
# ==============================================================================
oof_models = {
    "GPU_RandomForest": oof_rf,
    "GPU_LightGBM": oof_lgb,
    "GPU_XGBoost": oof_xgb,
    "GPU_CatBoost": oof_cat,
}
pred_models = {
    "GPU_RandomForest": pred_rf,
    "GPU_LightGBM": pred_lgb,
    "GPU_XGBoost": pred_xgb,
    "GPU_CatBoost": pred_cat,
}

model_names = list(oof_models.keys())
oof_ranks = np.column_stack([pd.Series(oof_models[m]).rank(pct=True).to_numpy(dtype=np.float32) for m in model_names])
pred_ranks = np.column_stack([pd.Series(pred_models[m]).rank(pct=True).to_numpy(dtype=np.float32) for m in model_names])

oof_ranks_t = torch.from_numpy(oof_ranks).to(TORCH_DEVICE)
y_train_t = torch.from_numpy(y_train.astype(np.float32)).to(TORCH_DEVICE)

def fast_gpu_auc(y_true, y_pred):
    order = torch.argsort(y_pred)
    y_sorted = y_true[order]
    n_pos = torch.sum(y_sorted)
    n_neg = y_sorted.numel() - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    ranks = torch.arange(1, y_sorted.numel() + 1, device=y_pred.device, dtype=torch.float64)
    sum_pos = torch.sum(ranks * y_sorted.to(torch.float64))
    return float(((sum_pos - n_pos * (n_pos + 1.0) / 2.0) / (n_pos * n_neg)).item())

def ensemble_objective(trial):
    weights = np.array([trial.suggest_float(f"w_{m}", 0.0, 1.0) for m in model_names], dtype=np.float32)
    if weights.sum() == 0:
        return 0.0
    weights /= weights.sum()
    with torch.no_grad():
        w_t = torch.from_numpy(weights).to(TORCH_DEVICE)
        pred_blend = torch.mv(oof_ranks_t, w_t)
        return fast_gpu_auc(y_train_t, pred_blend)

study = optuna.create_study(direction="maximize")
study.optimize(ensemble_objective, n_trials=250)

raw_w = np.array([study.best_params[f"w_{m}"] for m in model_names], dtype=np.float32)
best_weights = raw_w / raw_w.sum()

print("\n--- OPTIMAL GPU ENSEMBLE WEIGHTS ---")
for name, w in zip(model_names, best_weights):
    print(f"{name:20s}: {w:.4f} (Individual AUC: {roc_auc_score(y_train, oof_models[name]):.5f})")

final_oof = np.dot(oof_ranks, best_weights)
final_test = np.dot(pred_ranks, best_weights)

print("\n" + "=" * 72)
print(f">>> FINAL MULTI-MODEL ENSEMBLE OOF ROC-AUC: {roc_auc_score(y_train, final_oof):.5f} <<<")
print("=" * 72)


# ==============================================================================
# CELL 8: VERIFIED KAGGLE SUBMISSION GENERATION
# ==============================================================================
submission = pd.DataFrame({
    "order_line_id": submission_ids,
    "returned": np.clip(final_test, 1e-6, 1.0 - 1e-6)
})

# Verify exact order alignment with sample_submission.csv
submission = sample_sub_raw[["order_line_id"]].merge(submission, on="order_line_id", how="left")
assert len(submission) == 479995, f"Expected 479,995 rows, got {len(submission)}"
assert submission["returned"].isna().sum() == 0, "Submission contains NaN values!"

submission.to_csv("submission8.csv", index=False)
print("\n'submission8.csv' generated successfully! Summary Statistics:")
print(submission["returned"].describe())

PyTorch Device  : cuda (NVIDIA GeForce RTX 4060 Laptop GPU)
XGBoost / RF    : CUDA (GPU Hist Engine)
CatBoost Device : GPU
LightGBM Params : {'n_jobs': -1}
Working Directory: C:\Users\Vishesh\OneDrive\coding\Python\optiml
Loaded train.csv : (1400005, 27)
Loaded test.csv  : (479995, 26)
Auxiliary tables cleaned and transformed with stationary macroeconomic metrics.

Processing combined transaction table: (1880000, 26)
Extracting domain-specific return signals (bracketing, financial friction, landed cost)...

Encoding 21 categorical features with OrdinalEncoder...
Generating leak-free Out-Of-Fold Target Encodings across 8 attributes...
Final Feature Matrix: 144 features across 1,400,005 training lines.
Temporal Train (Months 1–20) : 1,216,163 lines
Temporal Valid (Months 21–24): 183,842 lines

--- Running GPU Optuna Tuning on Temporal Holdout (8 trials) ---
Best Temporal Holdout ROC-AUC: 0.50294

STARTING 5-FOLD GPU TRAINING ACROSS 4 DIVERSE ARCHITECTURES

--- Fold 1/5 ---


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 1 OOF AUCs -> RF: 0.50517 | LGB: 0.50311 | XGB: 0.50247 | CAT: 0.50130

--- Fold 2/5 ---


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 2 OOF AUCs -> RF: 0.50403 | LGB: 0.50197 | XGB: 0.50425 | CAT: 0.50278

--- Fold 3/5 ---


...

Default metric period is 5 because AUC is/are not implemented for GPU


Fold 4 OOF AUCs -> RF: 0.50043 | LGB: 0.50347 | XGB: 0.50322 | CAT: 0.50288

--- Fold 5/5 ---


Default metric period is 5 because AUC is/are not implemented for GPU


Fold 5 OOF AUCs -> RF: 0.50560 | LGB: 0.50353 | XGB: 0.50425 | CAT: 0.50288

FULL OUT-OF-FOLD ROC-AUC SUMMARY
Model 1 [GPU Random Forest] : 0.50336
Model 2 [GPU LightGBM]      : 0.50293
Model 3 [GPU XGBoost]       : 0.50329
Model 4 [GPU CatBoost]      : 0.50043

--- OPTIMAL GPU ENSEMBLE WEIGHTS ---
GPU_RandomForest    : 0.3174 (Individual AUC: 0.50336)
GPU_LightGBM        : 0.3261 (Individual AUC: 0.50293)
GPU_XGBoost         : 0.3526 (Individual AUC: 0.50329)
GPU_CatBoost        : 0.0038 (Individual AUC: 0.50043)

>>> FINAL MULTI-MODEL ENSEMBLE OOF ROC-AUC: 0.50431 <<<


AssertionError: Submission contains NaN values!